# Credit Card Approval Prediction - Model Training

Complete ML pipeline: preprocessing, training, evaluation, and model saving.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('.')))
if not os.path.exists(os.path.join(BASE_DIR, 'app.py')):
    BASE_DIR = os.path.abspath('..')
sys.path.insert(0, BASE_DIR)

from utils.preprocess import DataPreprocessor
from utils.helper import *
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print(f'Project root: {BASE_DIR}')

## 1. Load Dataset

In [ ]:
df = pd.read_csv(os.path.join(BASE_DIR, 'dataset', 'credit_card.csv'))
print(f'Records: {len(df)}, Columns: {len(df.columns)}')
df.head()

In [ ]:
df.info()
df.describe()

## 2. Class Distribution

In [ ]:
plot_class_distribution(df)
df['Approval_Status'].value_counts().plot(kind='bar', color=['#28a745', '#dc3545'])
plt.title('Approval Status Distribution')
plt.show()

## 3. Preprocessing & Train-Test Split

In [ ]:
preprocessor = DataPreprocessor()
X_train, X_test, y_train, y_test = preprocessor.fit(df)
feature_names = list(X_train.columns)
print(f'Training: {len(X_train)}, Test: {len(X_test)}')
print(f'Features: {feature_names}')

## 4. Train Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                            random_state=42, eval_metric='logloss'),
}

all_metrics = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test, name)
    all_metrics.append(metrics)
    trained_models[name] = model
    print(f'{name}: Accuracy={metrics["accuracy"]}, F1={metrics["f1_score"]}, AUC={metrics["roc_auc"]}')

## 5. Model Comparison

In [ ]:
comparison = pd.DataFrame(all_metrics)[
    ['model_name', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
]
comparison

In [ ]:
plot_accuracy_comparison(all_metrics)
plot_model_performance(all_metrics)
best = select_best_model(all_metrics)
print(f'Best Model: {best["model_name"]} (F1: {best["f1_score"]})')

## 6. Best Model Evaluation Charts

In [ ]:
best_model = trained_models[best['model_name']]
y_proba = best_model.predict_proba(X_test)[:, 1]

plot_confusion_matrix(best['confusion_matrix'], best['model_name'])
plot_roc_curve(y_test, y_proba, best['model_name'])
plot_feature_importance(best_model, feature_names, best['model_name'])

## 7. Save Best Model

In [ ]:
save_model_artifact(best_model, preprocessor, best, all_metrics, feature_names)
save_metrics_json(all_metrics, get_dataset_overview(df))
print('Model and metrics saved successfully!')